# preTextAnalysis — De PDFs a un corpus analítico

Este notebook parte de **planes de gobierno almacenados como archivos PDF en una carpeta de GitHub**.

La unidad de análisis será cada **plan de gobierno**. El objetivo de `preTextAnalysis` no es evaluar las hipótesis, sino construir de manera transparente la representación textual que utilizará `TextAnalysis.ipynb`.

Flujo:

**PDFs en GitHub → descarga → extracción de texto → una fila por plan → normalización/tokenización → corpus preparado**

## preTEXT-01 — Definir dónde están los PDFs

Edite solamente la URL base para que apunte a la carpeta *raw* de GitHub donde se encuentran los planes. Los nombres de archivo deben coincidir con los PDFs del repositorio.

> En GitHub no usamos la URL de la página que muestra el PDF, sino la URL `raw.githubusercontent.com` que entrega el archivo directamente.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# CAMBIE ESTA URL por la carpeta real de su repositorio.
RAW_BASE = "https://raw.githubusercontent.com/USUARIO/REPOSITORIO/main/planes_gobierno"

planes = pd.DataFrame([
    {"id_partido":"Partido_A", "partido":"Renovación Popular Social", "candidato":"Candidata A", "familia_ideologica":"izquierda", "tipo_texto":"ancla", "puntaje_referencia":-1.0, "archivo_pdf":"Partido_A_plan_gobierno.pdf"},
    {"id_partido":"Partido_B", "partido":"Frente Progresista", "candidato":"Candidato B", "familia_ideologica":"centro_izquierda", "tipo_texto":"virgin", "puntaje_referencia":np.nan, "archivo_pdf":"Partido_B_plan_gobierno.pdf"},
    {"id_partido":"Partido_C", "partido":"Movimiento Democrático", "candidato":"Candidata C", "familia_ideologica":"centro_izquierda", "tipo_texto":"virgin", "puntaje_referencia":np.nan, "archivo_pdf":"Partido_C_plan_gobierno.pdf"},
    {"id_partido":"Partido_D", "partido":"Centro Cívico", "candidato":"Candidato D", "familia_ideologica":"centro", "tipo_texto":"virgin", "puntaje_referencia":np.nan, "archivo_pdf":"Partido_D_plan_gobierno.pdf"},
    {"id_partido":"Partido_E", "partido":"Alianza Nacional", "candidato":"Candidata E", "familia_ideologica":"centro", "tipo_texto":"virgin", "puntaje_referencia":np.nan, "archivo_pdf":"Partido_E_plan_gobierno.pdf"},
    {"id_partido":"Partido_F", "partido":"Libertad Republicana", "candidato":"Candidato F", "familia_ideologica":"centro_derecha", "tipo_texto":"virgin", "puntaje_referencia":np.nan, "archivo_pdf":"Partido_F_plan_gobierno.pdf"},
    {"id_partido":"Partido_G", "partido":"Futuro Liberal", "candidato":"Candidata G", "familia_ideologica":"derecha", "tipo_texto":"virgin", "puntaje_referencia":np.nan, "archivo_pdf":"Partido_G_plan_gobierno.pdf"},
    {"id_partido":"Partido_H", "partido":"Mercado y Libertad", "candidato":"Candidato H", "familia_ideologica":"derecha", "tipo_texto":"ancla", "puntaje_referencia":1.0, "archivo_pdf":"Partido_H_plan_gobierno.pdf"},
])

planes["url_pdf"] = RAW_BASE.rstrip("/") + "/" + planes["archivo_pdf"]
planes

La tabla anterior contiene **metadata**, no el texto de los planes. El contenido que analizaremos todavía permanece dentro de los PDFs.

Los puntajes `-1` y `+1` identifican los dos textos de referencia que utilizaremos posteriormente en Wordscores. Los demás planes son textos vírgenes cuya posición será estimada.

## preTEXT-02 — Descargar los planes desde GitHub

Cada PDF se descarga a una carpeta local de trabajo. Así podemos conservar una copia exacta del documento que entró al análisis.

In [ ]:
import requests

carpeta_pdf = Path("planes_descargados")
carpeta_pdf.mkdir(exist_ok=True)

def descargar_pdf(url, destino):
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    if not r.content.startswith(b"%PDF"):
        raise ValueError(f"El archivo descargado desde {url} no parece ser un PDF.")
    destino.write_bytes(r.content)
    return destino

rutas = []
for _, fila in planes.iterrows():
    destino = carpeta_pdf / fila["archivo_pdf"]
    descargar_pdf(fila["url_pdf"], destino)
    rutas.append(str(destino))

planes["ruta_local"] = rutas
print("PDFs descargados:", len(planes))
planes[["id_partido", "archivo_pdf", "ruta_local"]]

## preTEXT-03 — Extraer el texto de cada PDF

Ahora abrimos cada PDF y recuperamos el texto de todas sus páginas. En este ejercicio los PDFs contienen texto digital. Si los documentos reales fueran imágenes escaneadas, este paso requeriría OCR y debería documentarse como una decisión adicional de preparación.

In [ ]:
import fitz  # PyMuPDF

def extraer_texto_pdf(ruta):
    doc = fitz.open(ruta)
    paginas = [pagina.get_text("text") for pagina in doc]
    return "\n".join(paginas).strip(), len(doc)

extraidos = planes["ruta_local"].apply(extraer_texto_pdf)
planes["texto"] = extraidos.str[0]
planes["n_paginas"] = extraidos.str[1]
planes["n_caracteres"] = planes["texto"].str.len()
planes["n_palabras_original"] = planes["texto"].str.split().str.len()

planes[["id_partido", "n_paginas", "n_caracteres", "n_palabras_original"]]

Antes de limpiar, revise si todos los documentos produjeron texto. Un PDF puede abrir correctamente y, sin embargo, no contener una capa de texto utilizable.

In [ ]:
assert (planes["n_caracteres"] > 0).all(), "Al menos un PDF no produjo texto."

for i in [0, 3, 7]:
    print("=" * 80)
    print(planes.loc[i, "id_partido"], "—", planes.loc[i, "partido"])
    print(planes.loc[i, "texto"][:500])

## preTEXT-04 — Limpiar artefactos del documento

La extracción incluye elementos que pertenecen al PDF pero que no necesariamente forman parte del argumento programático: títulos de portada, avisos pedagógicos y saltos de línea. En una investigación real también podrían aparecer números de página, encabezados, pies de página o índices.

Aquí retiramos únicamente elementos conocidos de nuestros documentos ficticios. **No existe una limpieza universal de PDFs.**

In [ ]:
import re

def limpiar_extraccion(texto):
    texto = re.sub(r"PLAN DE GOBIERNO", " ", texto, flags=re.I)
    texto = re.sub(r"Documento ficticio elaborado exclusivamente para fines pedagógicos\.?", " ", texto, flags=re.I)
    texto = re.sub(r"Candidatura:\s*[^\n]+", " ", texto, flags=re.I)
    texto = re.sub(r"Diagnóstico y prioridades|Economía y empleo|Políticas sociales|Instituciones y territorio", " ", texto, flags=re.I)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

planes["texto_extraido"] = planes["texto"].apply(limpiar_extraccion)
planes[["id_partido", "texto_extraido"]].head(2)

## preTEXT-05 — Normalizar y tokenizar

Para los análisis del capítulo utilizaremos una representación **Bag of Words**. Convertimos a minúsculas, retiramos puntuación y stopwords muy frecuentes y conservamos tokens alfabéticos de más de dos caracteres.

Esta es una decisión de representación, no una operación neutral: aquello que eliminamos ya no podrá contribuir a TF–IDF ni a Wordscores.

In [ ]:
stopwords_es = {
    "el","la","los","las","un","una","unos","unas","y","e","o","u",
    "de","del","al","a","en","para","por","con","sin","sobre","que",
    "se","su","sus","nuestro","nuestra","nuestros","nuestras","es","son",
    "ser","como","cuando","más","muy","también","cada","entre","desde",
    "todo","toda","todos","todas"
}

def preparar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r"[^a-záéíóúüñ\s]", " ", texto)
    tokens = [t for t in texto.split() if len(t) > 2 and t not in stopwords_es]
    return tokens

planes["tokens"] = planes["texto_extraido"].apply(preparar_texto)
planes["texto_limpio"] = planes["tokens"].str.join(" ")
planes["n_tokens"] = planes["tokens"].str.len()

planes[["id_partido", "n_palabras_original", "n_tokens"]]

## preTEXT-06 — Comparar PDF, extracción y representación

Antes de guardar el corpus, compare las distintas capas. El PDF es la fuente documental; `texto_extraido` es el texto recuperado y depurado de artefactos conocidos; `texto_limpio` es la representación que entrará a los algoritmos.

In [ ]:
for i in [0, 3, 7]:
    print("=" * 80)
    print(planes.loc[i, "id_partido"])
    print("\nEXTRAÍDO DEL PDF:\n", planes.loc[i, "texto_extraido"][:350])
    print("\nBAG OF WORDS PREPARADO:\n", planes.loc[i, "texto_limpio"][:350])

## preTEXT-07 — Guardar el corpus preparado

El archivo final contiene una fila por plan y conserva tanto metadata como el texto extraído y la representación preparada. Ese archivo constituye la entrada de `TextAnalysis.ipynb`.

In [ ]:
salida = Path("planes_gobierno_preparados.csv")

columnas = [
    "id_partido", "partido", "candidato", "familia_ideologica",
    "tipo_texto", "puntaje_referencia", "archivo_pdf", "url_pdf",
    "n_paginas", "n_caracteres", "n_palabras_original", "n_tokens",
    "texto_extraido", "texto_limpio"
]

planes[columnas].to_csv(salida, index=False)
print(f"Archivo guardado: {salida.resolve()}")
print("Filas:", len(planes), "| Columnas:", len(columnas))

`planes_gobierno_preparados.csv` es la entrada de `TextAnalysis.ipynb`.

El flujo completo fue:

**GitHub PDFs → descarga → extracción → revisión → limpieza documental → Bag of Words → CSV analítico**